# AI Programming — Lecture 21
## Lab 2: BLIP Image Captioning & VQA

CLIP/SigLIP이 이미지와 텍스트를 **matching**하는 데 초점을 둔다면,
BLIP은 이미지에 기반하여 **텍스트를 생성**할 수 있습니다.

이번 실습에서는 사전학습된 BLIP으로 두 가지 task를 수행합니다.

1. **Image Captioning**
2. **Visual Question Answering (VQA)**

### 학습 목표

- Pretrained BLIP model을 불러올 수 있습니다.
- 이미지로부터 자연어 caption을 생성할 수 있습니다.
- 이미지와 질문을 함께 입력하여 answer를 생성할 수 있습니다.
- Alignment 중심 VLM과 generative VLM의 차이를 이해합니다.

> 별도의 fine-tuning 없이 pretrained model의 inference만 사용합니다.

## 1. 필요한 패키지 설치

In [ ]:
!pip install -q -U transformers pillow requests matplotlib

## 2. 라이브러리와 실행 장치 설정

In [ ]:
import requests
from PIL import Image
import matplotlib.pyplot as plt

import torch

from transformers import (
    BlipProcessor,
    BlipForConditionalGeneration,
    BlipForQuestionAnswering,
)

device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("PyTorch version:", torch.__version__)
print("Device:", device)

## 3. 이미지 불러오기

필요하면 `image_url`만 바꾸어 다른 이미지로 실습할 수 있습니다.

In [ ]:
# 기본 예시 이미지
image_url = (
    "https://images.pexels.com/"
    "photos/20620196/pexels-photo-20620196.jpeg"
)

image = Image.open(
    requests.get(
        image_url,
        stream=True
    ).raw
).convert("RGB")

plt.figure(figsize=(6, 6))
plt.imshow(image)
plt.axis("off")
plt.title("Input image")
plt.show()

# Part I. Image Captioning

## 4. Pretrained Captioning Model 불러오기

사용 모델:

```text
Salesforce/blip-image-captioning-base
```

이미지를 입력하면 BLIP이 자연어 caption을 생성합니다.

In [ ]:
caption_model_name = (
    "Salesforce/blip-image-captioning-base"
)

caption_processor = (
    BlipProcessor.from_pretrained(
        caption_model_name
    )
)

caption_model = (
    BlipForConditionalGeneration
    .from_pretrained(
        caption_model_name
    )
    .to(device)
)

caption_model.eval()

print(
    "Loaded captioning model:",
    caption_model_name
)

## 5. Caption 생성

In [ ]:
inputs = caption_processor(
    images=image,
    return_tensors="pt"
).to(device)

with torch.no_grad():
    out = caption_model.generate(
        **inputs,
        max_new_tokens=30
    )

caption = caption_processor.decode(
    out[0],
    skip_special_tokens=True
)

print("Generated caption:")
print(caption)

### 확인할 내용

CLIP은 candidate text와의 similarity를 계산했지만,
BLIP captioning은 **새로운 token sequence를 생성**합니다.

```text
CLIP
image + candidate text → similarity

BLIP
image → generated text
```

# Part II. Visual Question Answering

## 6. Pretrained VQA Model 불러오기

사용 모델:

```text
Salesforce/blip-vqa-base
```

입력:

```text
Image + Question
```

출력:

```text
Answer
```

In [ ]:
vqa_model_name = (
    "Salesforce/blip-vqa-base"
)

vqa_processor = (
    BlipProcessor.from_pretrained(
        vqa_model_name
    )
)

vqa_model = (
    BlipForQuestionAnswering
    .from_pretrained(
        vqa_model_name
    )
    .to(device)
)

vqa_model.eval()

print(
    "Loaded VQA model:",
    vqa_model_name
)

## 7. 여러 질문하기

In [ ]:
questions = [
    "What is the person doing?",
    "How many dogs are there?",
    "What is on the beach?",
]

for question in questions:
    inputs = vqa_processor(
        images=image,
        text=question,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        out = vqa_model.generate(
            **inputs,
            max_new_tokens=20
        )

    answer = vqa_processor.decode(
        out[0],
        skip_special_tokens=True
    )

    print(
        f"Question: {question}"
    )

    print(
        f"Answer  : {answer}"
    )

    print("-" * 50)

## 8. 직접 해보기

이미지에 맞는 질문을 직접 작성해 보세요.

예:

```text
What color is the ...?
How many ... are there?
Where is the ...?
What is the person doing?
What is in front of the ...?
```

### 확인할 내용

- 이미지에 답이 명확하게 보이는 질문
- 이미지에 정보가 없는 질문
- 수량을 묻는 질문
- 행동을 묻는 질문

에서 결과가 어떻게 달라지는지 비교하세요.

# 정리

### CLIP / SigLIP

```text
Image ↔ Text
→ Alignment / Matching
```

### BLIP

```text
Image
→ Caption

Image + Question
→ Answer
```

즉, BLIP은 image-text alignment를 넘어
**image-grounded language generation**을 경험할 수 있는 VLM입니다.